# GC Example 09: APOGEE DR17 Chemistry

**EPS Research — Milky Way GC Corpus v1.3.1**

APOGEE DR17 provides mean [Fe/H] from H-band spectroscopy for 72 clusters.
Compare APOGEE vs Harris metallicities.

**Corpus:** Flynn (2026), Zenodo DOI: 10.5281/zenodo.19907766  
**Sources:** Harris (1996/2010), Vasiliev & Baumgardt (2021), Baumgardt et al. (2023), Schiavon et al. (2024) APOGEE DR17  
**Dependencies:** Python 3, numpy, matplotlib

In [ ]:
# ── Colab setup: auto-download corpus from Zenodo ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    import urllib.request
    CORPORA = {
        'harris_gc_corpus_v1.3.1.jsonl': 'https://zenodo.org/records/19907766/files/harris_gc_corpus_v1.3.1.jsonl',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")
    print("Ready.")
else:
    print("Running locally — corpus files loaded from working directory.")


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

clusters = []
with open('harris_gc_corpus_v1.3.1.jsonl') as f:
    for line in f:
        clusters.append(json.loads(line))
print(f"Total clusters: {len(clusters)}")


In [ ]:
data=[]
for c in clusters:
    if c.get('apogee_dr17') and c['apogee_dr17'].get('feh_apogee') is not None:
        if c.get('metallicity') and c['metallicity'].get('feh') is not None:
            data.append({'cluster':c['cluster_id'],
                         'feh_apogee':c['apogee_dr17']['feh_apogee'],
                         'feh_harris':c['metallicity']['feh'],
                         'n_members':c['apogee_dr17'].get('n_members',0)})
feh_a=[d['feh_apogee'] for d in data]; feh_h=[d['feh_harris'] for d in data]
diff=np.array(feh_a)-np.array(feh_h)
print(f"Clusters with both APOGEE and Harris [Fe/H]: {len(data)}")
print(f"Mean offset (APOGEE - Harris): {np.mean(diff):.3f} dex")
print(f"Std: {np.std(diff):.3f} dex")
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].scatter(feh_h,feh_a,s=20,alpha=0.7,color='#2166ac')
lim=[min(feh_h)-0.1,max(feh_h)+0.1]
axes[0].plot(lim,lim,'k--',lw=1,alpha=0.5,label='1:1')
axes[0].set_xlabel('Harris [Fe/H]',fontsize=11); axes[0].set_ylabel('APOGEE [Fe/H]',fontsize=11)
axes[0].set_title('APOGEE vs Harris Metallicity',fontsize=10); axes[0].legend()
axes[1].hist(diff,bins=15,color='#d62728',alpha=0.8,edgecolor='white')
axes[1].axvline(0,color='black',ls='--',lw=1.5)
axes[1].set_xlabel('APOGEE - Harris [Fe/H]',fontsize=11); axes[1].set_title('Offset Distribution',fontsize=10)
plt.suptitle('APOGEE DR17 vs Harris Metallicity\nEPS Research GC Corpus v1.3.1',fontsize=11)
plt.tight_layout(); plt.savefig('gc09_apogee_chemistry.png',dpi=150,bbox_inches='tight'); plt.show()